# Chunking Strategies

**Why chunk at all?** Embeddings degrade on long text (one vector must average everything),
and LLMs have limited context. Chunks are the unit both sides consume.

Three families compared in this notebook:

| Family | Decides boundary by | Cells |
|---|---|---|
| Fixed-size | character count | char-split, recursive-split |
| Structure-aware | document format (headers) | md-header, md-text |
| Context-aware | linguistic/semantic units | nltk (sentences) |

> Rule of thumb: no strategy wins universally - chunk size affects retrieval precision
(small = precise match, big = more context). Evaluate, don't guess.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]="Rag-basic"

### Fixed-size Chunking

Splits by character count only - ignores meaning and document structure.
Two flavors below: single-separator (`.`) vs recursive fallback chain.

Same PDF loader as `rag_basic` - one `Document` per page. Nothing new here.

In [ ]:
# PDF
from typing import List

import pypdf
from langchain_core.documents import Document

def load_pdf(file_path: str)-> List[Document]:
    reader = pypdf.PdfReader(file_path)
    docs = []
    for i, page in enumerate(reader.pages):
        docs.append(
            Document(
                page_content=page.extract_text() or "",
                metadata={"source":file_path, "page":i},
            )
        )

    return docs

docs = load_pdf(r"/path")
len(docs)

### Task 1 - CharacterTextSplitter (naive)

- Splits ONLY on the given `separator` (`.` = sentence-ish ends).
- If a "sentence" still exceeds `chunk_size`, it gets force-cut mid-text.
- Fails on text without periods (tables, code, lists) -> chunks hit the size limit and truncate.

In [ ]:
# Character Split

from langchain_text_splitters import CharacterTextSplitter

def chunk_chars(docs: List[Document])-> List:
    splitter = CharacterTextSplitter(
        separator='.',
        chunk_size=500,
        chunk_overlap=50,
        add_start_index=True
    )

    all_split = splitter.split_documents(docs)
    return all_split

all_char_splits=chunk_chars(docs)
all_char_splits[0].page_content
# len(all_char_splits)

### Task 2 - RecursiveCharacterTextSplitter (default choice)

- Tries separators in order: paragraphs -> lines -> sentences -> words -> characters.
- Keeps natural units intact whenever possible, force-cuts only as a last resort.
- Same size params as above -> compare outputs side-by-side with Task 1.

In [ ]:
# Recursive Character Split

from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_recursive_chars(docs: List[Document])-> List:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        add_start_index=True
    )

    all_split = splitter.split_documents(docs)
    return all_split

all_rc_splits=chunk_recursive_chars(docs)
all_rc_splits[0].page_content
# len(all_rc_splits)

### Task 3 - MarkdownHeaderTextSplitter (structure-aware)

- Splits at markdown HEADERS (`#`/`##`/`###`), never mid-section.
- Bonus: each chunk's metadata records its header path (`Header 1`, `Header 2`) -
free context that boosts retrieval filtering later.
- Note: uses `.split_text()` (string in, Documents out via header metadata) - not `.split_documents()`.
- Requires structured input: works great on the MD file, useless on plain text.

In [ ]:
# Markdown Header Split

from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_community.document_loaders import TextLoader

def chunk_md(file_path: str) -> List[Document]:

    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]

    loader = TextLoader(file_path)
    docs = loader.load()
    print(docs)
    md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)

    all_split = md_splitter.split_text(docs[0].page_content)
    return all_split

all_md_split=chunk_md(r"/path")
all_md_split

### Task 4 - MarkdownTextSplitter

- Size-based like Task 2, but its separator list knows markdown syntax.
- Different trade-off vs Task 3: respects size limits but loses the header-path metadata.
- Header splitter = structural fidelity; this one = predictable chunk sizes.

In [ ]:
# Markdown Text split

from langchain_text_splitters import MarkdownTextSplitter
from langchain_community.document_loaders import TextLoader

def chunk_md(file_path: str) -> List[Document]:

    loader = TextLoader(file_path)
    docs = loader.load()

    md_splitter = MarkdownTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    all_split = md_splitter.split_documents(docs)
    return all_split

all_md_split=chunk_md(r"/path")
all_md_split

## Context-Aware Chunking

Lets the *content itself* decide boundaries instead of raw character counts.
Below: sentence-aware splitting (NLTK) and web-page ingestion.

### Task 5 - NLTKTextSplitter (sentence-aware)

- Uses an NLP tokenizer model to detect real sentence boundaries (abbreviations, decimals, etc.),
then packs whole sentences up to `chunk_size` (default 4000).
- Middle ground: smarter than char-splitting, cheaper than embedding-based semantic chunking.
- First run downloads the punkt tokenizer model.

In [ ]:
# NLTK Split

from langchain_text_splitters import NLTKTextSplitter


def chunk_nltk(docs):
    splitter = NLTKTextSplitter()
    all_split = splitter.split_documents(docs)

    return all_split

docs = load_pdf((r"/path"))
all_nltk_split = chunk_nltk(docs)
all_nltk_split

### Task 6 - Full web ingestion

End-to-end mini pipeline: URL -> `UnstructuredURLLoader` (fetches + extracts page text)
-> recursive split. Same splitter as Task 2, proving splitters are source-agnostic:
PDF, MD or HTML - once content is a `Document`, chunking is identical.

In [ ]:
# URL Load
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

urls = ["https://workat.tech/core-cs/tutorial/processes-and-threads-os-6iboki1s2y3t/"]
loader = UnstructuredURLLoader(urls=urls)
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    add_start_index=True
)

all_split = splitter.split_documents(docs)
all_split